# PUBG Weapon Groups

Gắn dataset PUBG, bật Internet và GPU T4/P100 rồi Run All. Một notebook chạy đủ năm CSV, một seed 42 và CI bootstrap theo trận cho các mô hình, TabM, TabICLv2, báo cáo và PowerPoint.

In [ ]:
REPO_URL = "https://github.com/kttt294/PUBG-Weapon-Groups.git"
SOURCE_COMMIT = "efab5bba6263ce5539d67407df37051b5bad9420"
SOURCE_BUNDLE_SHA256 = "5f07e41f1f00f5c7554c147c16bcedf80327c2d33e67dac5b56e1b700f6a24af"
SOURCE_BUNDLE_BASE64 = "..."


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

WORK = Path("/kaggle/working")
if not WORK.is_dir():
    raise RuntimeError("Chạy notebook trên Kaggle.")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
SCRATCH = WORK / f"pubg_work_{RUN_ID}"
SCRATCH.mkdir()
REPO = SCRATCH / "source"
RUN = SCRATCH / "study"
SAVED = WORK / "pubg_results" / RUN_ID

import base64
import hashlib
bundle_bytes = base64.b64decode(SOURCE_BUNDLE_BASE64)
assert hashlib.sha256(bundle_bytes).hexdigest() == SOURCE_BUNDLE_SHA256
bundle_path = SCRATCH / "source.bundle"
bundle_path.write_bytes(bundle_bytes)
subprocess.run(["git", "clone", str(bundle_path), str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == SOURCE_COMMIT
print("Commit", actual_commit)


In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-r", "requirements.txt", "-r", "requirements-modern.txt",
], cwd=REPO, check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chọn GPU T4 hoặc P100 trong phần Accelerator rồi chạy lại.")
print("GPU", torch.cuda.get_device_name(0))


In [ ]:
filenames = [f"kill_match_stats_final_{i}.csv" for i in range(5)]
found = sorted(Path("/kaggle/input").rglob(filenames[0]))
if len(found) != 1:
    raise RuntimeError("Gắn đúng một bản dataset PUBG Match Deaths and Statistics.")
sources = [found[0].parent / name for name in filenames]
if not all(path.is_file() for path in sources):
    raise FileNotFoundError("Thiếu một trong năm CSV sự kiện.")
free_gib = shutil.disk_usage(SCRATCH).free / 2**30
if free_gib < 14:
    raise RuntimeError(f"Chỉ còn {free_gib:.1f} GiB đĩa, cần ít nhất 14 GiB.")
config = json.loads((REPO / "configs" / "study.json").read_text())
print("Nguồn", len(sources), "CSV")
print("Seed mô hình", config["seeds"])
print("Phương pháp mới", "TabM và TabICLv2")
print("Đĩa trống", round(free_gib, 1), "GiB")


In [ ]:
environment = {
    **os.environ,
    "HF_HOME": str(SCRATCH / "huggingface"),
    "OMP_NUM_THREADS": str(config["jobs"]),
    "MKL_NUM_THREADS": str(config["jobs"]),
    "OPENBLAS_NUM_THREADS": str(config["jobs"]),
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONUNBUFFERED": "1",
}
command = [
    sys.executable, "-u", "-m", "pubg_weapon.study",
    "--source", *map(str, sources), "--output", str(RUN),
]
try:
    process = subprocess.run(command, cwd=REPO, env=environment, check=False)
    (RUN / 'process_exit.json').write_text(json.dumps({'returncode': process.returncode}), encoding='utf-8')
finally:
    SAVED.mkdir(parents=True, exist_ok=True)
    if RUN.is_dir():
        for path in RUN.iterdir():
            target = SAVED / path.name
            if path.name == "prepared":
                target.mkdir()
                for artifact in path.iterdir():
                    if artifact.is_file() and artifact.suffix in (".json", ".csv"):
                        shutil.copy2(artifact, target / artifact.name)
            elif path.is_dir():
                shutil.copytree(path, target, ignore=shutil.ignore_patterns("*_cache"))
            else:
                shutil.copy2(path, target)
    subprocess.run([
        "git", "-C", str(REPO), "archive", "--format=zip",
        f"--output={SAVED / 'source.zip'}", "HEAD",
    ], check=True)
    shutil.make_archive(str(WORK / f"pubg_results_{RUN_ID}"), "zip", SAVED)
    print("Kết quả đã lưu", SAVED)

if process.returncode != 0:
    raise RuntimeError("Lượt chạy gặp lỗi. Kết quả đã hoàn tất và error.txt đã được giữ trong Output.")
status = json.loads((SAVED / "status.json").read_text())
assert status["status"] == "completed"
shutil.rmtree(SCRATCH)


In [ ]:
import pandas as pd
from IPython.display import Image, display

for name in ("full_summary.csv", "modern_summary.csv"):
    display(pd.read_csv(SAVED / "deliverables" / name))
display(Image(filename=str(SAVED / "deliverables" / "modern_comparison.png")))
print("Báo cáo", SAVED / "deliverables" / "bao_cao_khai_pha_du_lieu.pdf")
print("LaTeX", SAVED / "deliverables" / "main.tex")
print("PowerPoint", SAVED / "deliverables" / "Thuyet_trinh_PUBG.pptx")
print("Đối chiếu yêu cầu", (SAVED / "deliverables" / "requirements_check.json").read_text())
